# Fine-tuning RoBERTa

Same task as `01_baseline_tfidf.ipynb`, this time with `roberta-base` fine-tuned on a stratified 10k subset of the labeled reviews.
Run on Colab with a T4 GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
!pip install -q transformers datasets accelerate

import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to T4"
print(torch.cuda.get_device_name(0))

Tesla T4


## 1. Data

`reviews_roberta.csv` is exported at the end of the baseline notebook. I put it on Google Drive and read it from there.

In [ ]:
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split

drive.mount("/content/drive")

df = pd.read_csv("/content/drive/MyDrive/reviews_roberta.csv")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(len(train_df), len(test_df))

Mounted at /content/drive
8000 2000


## 2. Setup

Reviews are long (median around 300 words), so sequence length is one of the things tested below, along with the number of epochs.

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def make_datasets(max_length):
    def tokenize(batch):
        return tokenizer(batch["review"], truncation=True, max_length=max_length, padding="max_length")
    train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(tokenize, batched=True)
    test_ds = Dataset.from_pandas(test_df, preserve_index=False).map(tokenize, batched=True)
    return train_ds, test_ds


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }


def train(max_length, epochs):
    set_seed(42)
    train_ds, test_ds = make_datasets(max_length)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2,
        id2label={0: "negative", 1: "positive"},
        label2id={"negative": 0, "positive": 1},
    )

    # 512 tokens don't fit with batch 16 on a T4, so accumulate gradients instead
    batch_size = 16 if max_length <= 256 else 8

    args = TrainingArguments(
        output_dir=f"runs/{max_length}_{epochs}",
        num_train_epochs=epochs,
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=16 // batch_size,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        fp16=True,
        report_to="none",
        seed=42,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      eval_dataset=test_ds, compute_metrics=compute_metrics)
    trainer.train()
    return trainer

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## 3. Experiments

Three runs: 256 vs 512 tokens, then 2 vs 3 epochs at 512. Takes about 30 min in total on a T4.

In [ ]:
results = []

for max_length, epochs in [(256, 2), (512, 2), (512, 3)]:
    trainer = train(max_length, epochs)
    scores = trainer.evaluate()
    results.append({
        "max_length": max_length,
        "epochs": epochs,
        "accuracy": round(scores["eval_accuracy"], 3),
        "f1": round(scores["eval_f1"], 3),
        "val_loss": round(scores["eval_loss"], 3),
    })
    if (max_length, epochs) != (512, 3):
        del trainer
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.331347,0.174955,0.945000,0.966007,0.939303,0.994275
2,0.225910,0.183055,0.961000,0.975112,0.978233,0.972010
3,0.166966,0.193217,0.964000,0.977186,0.973485,0.980916


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.166966,0.193217,3,0.964000,0.977186,0.973485,0.980916


,max_length,epochs,accuracy,f1,val_loss
0,256,2,0.950,0.968,0.199
1,512,2,0.966,0.978,0.163
2,512,3,0.964,0.977,0.193


## 4. Results

Going from 256 to 512 tokens gives the biggest jump, which makes sense given how long the reviews are. A third epoch doesn't help: F1 stays flat and validation loss goes back up, so the model starts overfitting. I keep 512 tokens / 2 epochs.

For reference, the TF-IDF baseline reached F1 = 0.952.

## 5. A few examples

Including some where the model gets it wrong.

In [ ]:
from transformers import pipeline

# best config from the table above
trainer = train(512, 2)

clf = pipeline("sentiment-analysis", model=trainer.model, tokenizer=tokenizer, device=0)

examples = [
    "This anime was absolutely a masterpiece, I cried at the ending.",
    "Not boring at all, actually it was amazing.",
    "I really liked this anime, but it was kinda shitty at the end.",
    "I'm quite doubting the fact that some people are liking this anime it's very unique but yeah i'll pass",
]

for text in examples:
    pred = clf(text, truncation=True, max_length=512)[0]
    print(f"{pred['label']:>8} ({pred['score']:.2f})  {text}")

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.329810,0.166913,0.959000,0.974327,0.959309,0.989822
2,0.195054,0.156883,0.966000,0.978399,0.977157,0.979644


positive (1.00)  This anime was absolutely a masterpiece, I cried at the ending.
positive (1.00)  Not boring at all, actually it was amazing.
negative (0.99)  I really liked this anime, but it was kinda shitty at the end.
positive (1.00)  I'm quite doubting the fact that some people are liking this anime it's very unique but yeah i'll pass


## 6. Saving the model

Saved to Drive, then pushed to the Hugging Face Hub for the Streamlit app.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

path = "/content/drive/MyDrive/anime-sentiment-roberta"
trainer.save_model(path)
tokenizer.save_pretrained(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/anime-sentiment-roberta/tokenizer_config.json',
 '/content/drive/MyDrive/anime-sentiment-roberta/tokenizer.json')